# Cleaning the Raw Data: California, 2016–2020

Produces `data/clean_ca_2016_2020.parquet`, which every other notebook loads with `src.data.load()`.
Run top to bottom with **Restart → Run All**.

In [ ]:
import sys; sys.path.append("..")
import glob
import re
import numpy as np
import pandas as pd

## 1. Load the 5 yearly files, keeping only California fires

In [ ]:
files = sorted(glob.glob("../data/*_FPA_FOD_cons.csv"))
print(files)

# Filter each file to California as it loads (keeps memory low)
df = pd.concat((pd.read_csv(f, low_memory=False).query("STATE == 'CA'") for f in files),
               ignore_index=True)
print(df.shape)
df["FIRE_YEAR"].value_counts().sort_index()

## 2. First look

In [ ]:
pd.set_option("display.max_rows", 400)
print(df["FIRE_SIZE_CLASS"].value_counts().sort_index())   # target balance
df.isna().mean().sort_values(ascending=False).head(30)     # most-missing columns

## 3. Drop leakage and ID columns

- **Leakage:** anything only known after the fire (containment, cause, MTBS/ICS-209 links that only exist for large fires).
- **`_5D_` columns:** the 5-day windows are centered on ignition (2 days before to 2 days after), so they include post-ignition weather (Pourmohamad et al., 2024).
- `FIRE_SIZE` is kept for analysis only, never as a model input.

In [ ]:
drop = [
    # Known only after the fire
    "CONT_DATE", "CONT_DOY", "CONT_TIME",
    "NWCG_CAUSE_CLASSIFICATION", "NWCG_GENERAL_CAUSE", "NWCG_CAUSE_AGE_CATEGORY",
    "MTBS_ID", "MTBS_FIRE_NAME", "ICS_209_PLUS_INCIDENT_JOIN_ID", "ICS_209_PLUS_COMPLEX_JOIN_ID",
    "COMPLEX_NAME",
    # IDs and names (no predictive meaning)
    "FOD_ID", "FPA_ID", "SOURCE_SYSTEM_TYPE", "SOURCE_SYSTEM", "NWCG_REPORTING_AGENCY",
    "NWCG_REPORTING_UNIT_ID", "NWCG_REPORTING_UNIT_NAME", "SOURCE_REPORTING_UNIT",
    "SOURCE_REPORTING_UNIT_NAME", "LOCAL_FIRE_REPORT_ID", "LOCAL_INCIDENT_ID",
    "FIRE_CODE", "FIRE_NAME",
]
drop += [c for c in df.columns if "_5D_" in c]   # post-ignition weather windows

df = df.drop(columns=drop, errors="ignore").copy()
print(df.shape)
print("Remaining _5D_ columns:", [c for c in df.columns if "_5D_" in c])   # should be []

## 4. Placeholder values and dates

In [ ]:
# Treat -999 / -9999 style placeholders as missing
num_cols = df.select_dtypes("number").columns
df[num_cols] = df[num_cols].mask(df[num_cols] <= -999)

df["DISCOVERY_DATE"] = pd.to_datetime(df["DISCOVERY_DATE"], errors="coerce")
df["DISCOVERY_MONTH"] = df["DISCOVERY_DATE"].dt.month

## 5. Missing values: drop only columns that are mostly empty (gaps get filled later, on train only)

In [ ]:
missing = df.isna().mean()
print(missing[missing > 0.5].sort_values(ascending=False))

df = df.loc[:, missing <= 0.5]
df = df.dropna(subset=["FIRE_SIZE_CLASS", "LATITUDE", "LONGITUDE"])
df.shape

## 6. Target: merge the rarest size classes

In [ ]:
merge = {"A": "A", "B": "B", "C": "C", "D": "D-E", "E": "D-E", "F": "F-G", "G": "F-G"}
df["TARGET"] = df["FIRE_SIZE_CLASS"].map(merge)
df["TARGET"].value_counts().sort_index()

## 7. Percentile columns: text ranges → numbers

Labels like `"<10th"`, `"10th-30th"`, `">90th"` become the midpoint of the range (5, 20, … 95), so they keep their order.

In [ ]:
pct_cols = [c for c in df.columns if c.endswith("_Percentile")]
print("Before:", df[pct_cols[0]].value_counts(dropna=False).to_dict())

def pct_to_number(v):
    if pd.isna(v):
        return np.nan
    if isinstance(v, (int, float, np.number)):
        return float(v)
    s = str(v).strip()
    nums = [float(x) for x in re.findall(r"\d+(?:\.\d+)?", s)]
    if not nums:
        return np.nan
    if s.startswith("<"):
        return nums[0] / 2              # "<10th"  -> 5
    if s.startswith(">"):
        return (nums[0] + 100) / 2      # ">90th"  -> 95
    return sum(nums) / len(nums)        # "10th-30th" -> 20

for c in pct_cols:
    df[c] = df[c].map(pct_to_number)

print("After:", df[pct_cols[0]].value_counts(dropna=False).sort_index().to_dict())

## 8. Category codes → one-hot columns

`Land_Cover` and `EVT` are codes, not amounts. Keep the 20 most common codes each, lump the rest into "other".

In [ ]:
cat_cols = [c for c in ["Land_Cover", "EVT"] if c in df.columns]
for c in cat_cols:
    top = df[c].value_counts().index[:20]
    df[c] = df[c].where(df[c].isin(top), "other").fillna("missing").astype(str)

df = pd.get_dummies(df, columns=cat_cols, prefix=["LC" if c == "Land_Cover" else "EVT" for c in cat_cols], dtype=float)
print([c for c in df.columns if c.startswith(("LC_", "EVT_"))])

## 9. Aspect is a compass direction → sine and cosine

In [ ]:
for c in ["Aspect", "Aspect_1km"]:
    if c in df.columns:
        rad = np.deg2rad(df[c])
        df[c + "_sin"], df[c + "_cos"] = np.sin(rad), np.cos(rad)

## 10. Regions for the location split

California is small, so use **0.5° × 0.5° grid blocks** (about 50 km) instead of 1°. Every block goes entirely into train or test.

In [ ]:
df["REGION"] = (np.floor(df["LATITUDE"] / 0.5).astype(int).astype(str) + "_" +
                np.floor(df["LONGITUDE"] / 0.5).astype(int).astype(str))

print(df["REGION"].nunique(), "regions")
# Are large fires spread across many regions, or concentrated in a few?
fg = df.groupby("REGION")["TARGET"].apply(lambda s: (s == "F-G").sum()).sort_values(ascending=False)
print((fg > 0).sum(), "regions contain at least one F-G fire")
fg.head(10)

## 11. Check the feature groups and the folds

In [ ]:
from src.data import feature_sets, cv_folds

sets = feature_sets(df)
for name, cols in sets.items():
    print(name, len(cols), "features")

# Class mix in each test fold (should be roughly similar; watch the F-G counts)
pd.DataFrame({f"fold {i}": te["TARGET"].value_counts()
              for i, (tr, te) in enumerate(cv_folds(df))}).sort_index()

## 12. Keep only what's needed and save

In [ ]:
extra = ["TARGET", "REGION", "FIRE_SIZE", "FIRE_SIZE_CLASS", "FIRE_YEAR", "STATE",
         "DISCOVERY_DATE", "Population", "GAP_Prity", "EPL_MOBILE"]
keep = list(dict.fromkeys(sets["model3_all"] + [c for c in extra if c in df.columns]))
df = df[keep].copy()
print(df.shape)

df.to_parquet("../data/clean_ca_2016_2020.parquet")

## 13. Sanity check: reload the saved file

In [ ]:
from src.data import load
check = load()
print(check.shape)
check["TARGET"].value_counts().sort_index()